# Kalshi Market Maker

A standalone notebook for posting two-sided limit quotes inside the Kalshi bid-ask spread.

## Strategy

For a chosen Kalshi market with bid-ask spread ≥ `min_spread_cents`:

1. Compute a **fair value** for YES probability — either from the existing v2 model (lognormal with live RV) or a simple Bayesian weighted-mid.
2. Post a **buy-YES limit** at `fair − half_spread`.
3. Post a **sell-YES limit** at `fair + half_spread` (Kalshi-equivalent: buy NO at `1 − (fair + half_spread)`).
4. When either fills, mark inventory.
5. **Cancel + replace** every `quote_refresh_sec` (default 30s) or when fair value moves > `quote_drift_cents`.
6. **Inventory caps**: stop adding to a side when it's already $X-loaded; widen quotes on the heavy side to encourage flattening.
7. **Time-exit**: cancel all quotes and flatten positions in the last `time_exit_min` minutes before settlement.

## Runtime modes

- `paper`: simulates fills against the live Kalshi orderbook (bid hit → assume YES bought at our quote price). No real orders.
- `live`: actual orders sent via `kalshi_live.place_order(..., type_="limit", yes_price=..., no_price=...)`.

## Risk

- Max inventory per market: $50 (about 100-200 contracts depending on entry price)
- Max total inventory across markets: $150
- Halt on cumulative session loss > $30
- Strict order ack flag for live mode


## 1. Setup

In [ ]:
import os, json, time, threading, requests, sqlite3
import numpy as np, pandas as pd
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Optional, Dict, List
from dateutil import parser as dtparser

import warnings; warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print(f"market-maker boot @ {datetime.now(timezone.utc).isoformat()}")


In [ ]:
import base64
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding


def load_credentials(env_path: str = "~/.kalshi/credentials.env") -> Dict[str, str]:
    creds = {}
    path = Path(env_path).expanduser()
    if not path.exists():
        print(f"warn: no credentials file at {path}")
        return creds
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        creds[k.strip()] = v.strip()
    return creds


class KalshiClient:
    DEMO_URL = "https://demo-api.kalshi.co/trade-api/v2"
    PROD_URL = "https://api.elections.kalshi.com/trade-api/v2"

    def __init__(self, env="prod", key_id=None, private_key_path=None):
        assert env in ("demo", "prod")
        self.env = env
        self.base_url = self.DEMO_URL if env == "demo" else self.PROD_URL
        self._path_prefix = "/trade-api/v2"
        creds = load_credentials()
        if env == "prod":
            self.key_id = key_id or creds.get("KALSHI_PROD_KEY_ID")
            kp = private_key_path or creds.get("KALSHI_PROD_PRIVATE_KEY_PATH")
        else:
            self.key_id = key_id or creds.get("KALSHI_DEMO_KEY_ID")
            kp = private_key_path or creds.get("KALSHI_DEMO_PRIVATE_KEY_PATH")
        self.private_key = None
        if kp:
            kp_path = Path(kp).expanduser()
            if kp_path.exists():
                with open(kp_path, "rb") as f:
                    self.private_key = serialization.load_pem_private_key(
                        f.read(), password=None)
        self.session = requests.Session()

    def _sign(self, method, path):
        if not self.private_key or not self.key_id:
            return {}
        ts = str(int(time.time() * 1000))
        path_no_query = path.split("?")[0]
        msg = (ts + method + path_no_query).encode("utf-8")
        sig = self.private_key.sign(
            msg,
            padding.PSS(mgf=padding.MGF1(hashes.SHA256()),
                         salt_length=padding.PSS.DIGEST_LENGTH),
            hashes.SHA256())
        return {
            "KALSHI-ACCESS-KEY":       self.key_id,
            "KALSHI-ACCESS-SIGNATURE": base64.b64encode(sig).decode("utf-8"),
            "KALSHI-ACCESS-TIMESTAMP": ts,
        }

    def _get(self, path, params=None):
        url = self.base_url + path
        h = self._sign("GET", self._path_prefix + path)
        r = self.session.get(url, headers=h, params=params, timeout=10)
        r.raise_for_status()
        return r.json()

    def _post(self, path, body):
        url = self.base_url + path
        h = self._sign("POST", self._path_prefix + path)
        h["Content-Type"] = "application/json"
        r = self.session.post(url, headers=h, json=body, timeout=10)
        r.raise_for_status()
        return r.json()

    def _delete(self, path):
        url = self.base_url + path
        h = self._sign("DELETE", self._path_prefix + path)
        r = self.session.delete(url, headers=h, timeout=10)
        r.raise_for_status()
        return r.json() if r.text else {}

    def get_markets(self, event_ticker=None, limit=200, status="open"):
        params = {"limit": limit, "status": status}
        if event_ticker: params["event_ticker"] = event_ticker
        return self._get("/markets", params)

    def get_market(self, ticker):
        return self._get(f"/markets/{ticker}")

    def get_orderbook(self, ticker, depth=10):
        return self._get(f"/markets/{ticker}/orderbook", {"depth": depth})

    def get_balance(self):
        return self._get("/portfolio/balance")

    def get_positions(self, ticker=None):
        params = {"ticker": ticker} if ticker else None
        return self._get("/portfolio/positions", params)

    def get_orders(self, ticker=None, status="resting"):
        params = {"status": status}
        if ticker: params["ticker"] = ticker
        return self._get("/portfolio/orders", params)

    def place_order(self, ticker, side, action, count, type_="limit",
                     yes_price=None, no_price=None,
                     expiration_ts=None, client_order_id=None):
        body = {
            "ticker":            ticker,
            "side":              side,         # "yes" or "no"
            "action":            action,       # "buy" or "sell"
            "count":             int(count),
            "type":              type_,        # "limit" or "market"
            "client_order_id":   client_order_id or f"mm-{int(time.time()*1000)}",
        }
        if type_ == "limit":
            if side == "yes" and yes_price is not None:
                body["yes_price"] = int(round(yes_price * 100))
            elif side == "no" and no_price is not None:
                body["no_price"] = int(round(no_price * 100))
        if expiration_ts is not None:
            body["expiration_ts"] = int(expiration_ts)
        return self._post("/portfolio/orders", body)

    def cancel_order(self, order_id):
        return self._delete(f"/portfolio/orders/{order_id}")


# Read-only client for market data (no signing required)
kalshi_md = KalshiClient(env="prod")
kalshi_md.private_key = None
kalshi_md.key_id      = None

# Auth client for orders (uses prod credentials)
kalshi_live = None
try:
    kalshi_live = KalshiClient(env="prod")
    if kalshi_live.private_key is None or kalshi_live.key_id is None:
        print("warn: no prod credentials; live mode disabled")
        kalshi_live = None
    else:
        bal = kalshi_live.get_balance()
        balance_cents = bal.get("balance", 0) if isinstance(bal, dict) else 0
        print(f"live auth OK. balance: ${balance_cents/100:.2f}")
except Exception as e:
    print(f"live auth failed: {e}")


## 2. Configuration

In [ ]:
MM_CFG = {
    # Strategy mode
    "mode":                       "paper",   # "paper" or "live"
    "i_acknowledge_real_money":   False,

    # Market selection
    "min_spread_cents":           3,         # only quote markets with spread >= this
    "max_ttl_hours":              4.0,       # only quote markets settling within this
    "min_ttl_min":                10,        # don't quote markets too close to settle
    "event_series_whitelist":    ("KXBTC","KXBTCD"),

    # Quote sizing
    "default_size_contracts":     20,        # contracts per side per quote
    "max_inventory_per_market":   50,        # halt adding to a side once $50 loaded
    "max_inventory_total":        150,       # total $ across all markets
    "min_size_per_quote":         5,         # minimum contracts; below this skip

    # Quote pricing
    "quote_offset_cents":         1,         # post 1c inside the existing top-of-book
    "fair_value_method":          "weighted_mid",   # "weighted_mid" or "lognormal"
    "max_quote_drift_cents":      2,         # cancel+replace if our fair moves this much

    # Refresh cadence
    "quote_refresh_sec":          30,        # cancel+replace every N seconds
    "loop_interval_sec":          5,         # main loop tick

    # Risk
    "session_loss_halt_usd":     -30.0,      # halt trading when session realized < this
    "kalshi_fee_per_contract":   0.007,      # ~$0.07/contract per side

    # Time exits
    "time_exit_min":              5,         # flatten + stop quoting in last 5 min

    # Persistence
    "db_path":                    "~/.btc_kalshi_bot/market_maker.db",
}

print("MM_CFG defined:")
for k, v in MM_CFG.items():
    print(f"  {k:30s} = {v}")


## 3. Local persistence (independent from main bot's paper_trades)

In [ ]:
_MM_DB_PATH = Path(MM_CFG["db_path"]).expanduser()
_MM_DB_PATH.parent.mkdir(parents=True, exist_ok=True)


def _mm_conn():
    c = sqlite3.connect(str(_MM_DB_PATH))
    c.row_factory = sqlite3.Row
    return c


def init_mm_db():
    conn = _mm_conn()
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS mm_quotes (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        ts              TEXT NOT NULL,
        ticker          TEXT NOT NULL,
        side            TEXT NOT NULL,    -- 'yes' or 'no'
        price           REAL NOT NULL,    -- our quote price
        contracts       INTEGER NOT NULL,
        order_id        TEXT,
        client_order_id TEXT,
        status          TEXT,             -- 'resting','filled','canceled','expired'
        filled_at       TEXT,
        canceled_at     TEXT,
        fair_value      REAL,
        market_yes_bid  REAL,
        market_yes_ask  REAL
    );
    CREATE INDEX IF NOT EXISTS idx_mm_quotes_ts ON mm_quotes(ts);
    CREATE INDEX IF NOT EXISTS idx_mm_quotes_ticker ON mm_quotes(ticker);

    CREATE TABLE IF NOT EXISTS mm_fills (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        ts              TEXT NOT NULL,
        ticker          TEXT NOT NULL,
        side            TEXT NOT NULL,
        action          TEXT NOT NULL,    -- 'buy' or 'sell'
        price           REAL NOT NULL,
        contracts       INTEGER NOT NULL,
        cost_dollars    REAL NOT NULL,
        fees_dollars    REAL NOT NULL,
        quote_id        INTEGER,
        FOREIGN KEY(quote_id) REFERENCES mm_quotes(id)
    );

    CREATE TABLE IF NOT EXISTS mm_inventory (
        ticker          TEXT PRIMARY KEY,
        net_yes         INTEGER NOT NULL DEFAULT 0,    -- + = long YES, - = short
        avg_cost_per    REAL,
        total_cost      REAL NOT NULL DEFAULT 0,
        last_updated    TEXT
    );
    """)
    conn.commit(); conn.close()
init_mm_db()
print(f"market-maker DB at {_MM_DB_PATH}")


## 4. Market discovery — find wide-spread BTC markets

In [ ]:
def find_quotable_markets(min_spread_cents=None, max_ttl_hours=None,
                             min_ttl_min=None, whitelist=None):
    """Return list of dicts for BTC markets with bid-ask spread >= threshold,
    sorted by spread descending."""
    if min_spread_cents is None:
        min_spread_cents = MM_CFG["min_spread_cents"]
    if max_ttl_hours is None:
        max_ttl_hours = MM_CFG["max_ttl_hours"]
    if min_ttl_min is None:
        min_ttl_min = MM_CFG["min_ttl_min"]
    if whitelist is None:
        whitelist = MM_CFG["event_series_whitelist"]

    now = datetime.now(timezone.utc)
    quotables = []

    # Iterate known BTC series; cheap and reliable
    for series in whitelist:
        try:
            resp = kalshi_md._get("/events",
                                    {"series_ticker": series,
                                     "status": "open", "limit": 200})
            events = resp.get("events", [])
        except Exception as e:
            print(f"  err scanning {series}: {e}")
            continue

        for ev in events:
            ev_tk = ev["event_ticker"]
            try:
                mkts = kalshi_md.get_markets(
                    event_ticker=ev_tk, limit=200).get("markets", [])
            except Exception:
                continue

            for m in mkts:
                yb = m.get("yes_bid"); ya = m.get("yes_ask")
                if yb is None or ya is None: continue
                yb_pct, ya_pct = float(yb), float(ya)
                spread_c = ya_pct - yb_pct
                if spread_c < min_spread_cents: continue

                ct = m.get("close_time")
                if not ct: continue
                close_t = dtparser.isoparse(ct)
                ttl_min = (close_t - now).total_seconds() / 60
                if not (min_ttl_min <= ttl_min <= max_ttl_hours * 60):
                    continue

                quotables.append({
                    "ticker":       m["ticker"],
                    "event":        ev_tk,
                    "series":       series,
                    "yes_bid":      yb_pct / 100,
                    "yes_ask":      ya_pct / 100,
                    "spread_cents": spread_c,
                    "ttl_min":      ttl_min,
                    "ttl_hours":    ttl_min / 60,
                    "close_time":   close_t,
                    "volume":       m.get("volume", 0),
                    "open_int":     m.get("open_interest", 0),
                    "floor":        m.get("floor_strike"),
                    "cap":          m.get("cap_strike"),
                    "last_price":   m.get("last_price"),
                })
    quotables.sort(key=lambda x: -x["spread_cents"])
    return quotables


# Quick demo
markets = find_quotable_markets()
print(f"found {len(markets)} quotable markets:")
for m in markets[:10]:
    print(f"  {m['ticker']:35s}  bid/ask=${m['yes_bid']:.2f}/${m['yes_ask']:.2f}  "
          f"spread={m['spread_cents']:.0f}c  ttl={m['ttl_hours']:.1f}h")


## 5. Fair value computation

In [ ]:
def fair_value_weighted_mid(market):
    """Bayesian weighted mid: just the simple midpoint, optionally biased by
    last_price if available. Returns a probability in [0, 1]."""
    yb, ya = market["yes_bid"], market["yes_ask"]
    mid = (yb + ya) / 2
    last = market.get("last_price")
    if last is not None:
        last_p = float(last) / 100
        # Weight toward last_price if it's recent enough to trust
        return 0.7 * mid + 0.3 * last_p
    return mid


def fair_value_lognormal(market, btc_spot, sigma_annualized):
    """Black-Scholes-style P(BTC > strike) using realized vol.
    Falls back to weighted mid if strike or spot unavailable."""
    floor = market.get("floor")
    if floor is None or btc_spot is None or sigma_annualized is None:
        return fair_value_weighted_mid(market)
    T_years = market["ttl_min"] / (60 * 24 * 365)
    if T_years <= 0 or sigma_annualized <= 0:
        return fair_value_weighted_mid(market)
    from scipy.stats import norm
    K = float(floor)
    d = (np.log(btc_spot / K) - 0.5 * sigma_annualized**2 * T_years) / \
         (sigma_annualized * np.sqrt(T_years))
    p_above = float(norm.cdf(d))
    # Cumulative T-strike: P(YES) = P(BTC > K)
    # Bucket B-strike: P(YES) = P(K_floor <= BTC < K_cap), need cap too
    cap = market.get("cap")
    if "-B" in market["ticker"] and cap is not None:
        d_cap = (np.log(btc_spot / float(cap)) - 0.5 * sigma_annualized**2 * T_years) / \
                 (sigma_annualized * np.sqrt(T_years))
        p_above_cap = float(norm.cdf(d_cap))
        return max(0.0, p_above - p_above_cap)
    return p_above


def get_btc_spot():
    try:
        r = requests.get("https://api.coinbase.com/v2/prices/BTC-USD/spot",
                          timeout=4)
        return float(r.json()["data"]["amount"])
    except Exception:
        return None


def compute_fair_value(market, method=None):
    method = method or MM_CFG["fair_value_method"]
    if method == "lognormal":
        spot = get_btc_spot()
        # Hardcode 50% annualized as a default; in production wire this to
        # a live RV estimator from a Coinbase tick stream.
        sigma = 0.50
        return fair_value_lognormal(market, spot, sigma)
    return fair_value_weighted_mid(market)


## 6. Quote engine — compute target bid + ask

In [ ]:
def compute_target_quotes(market, fair_value, inventory=0):
    """Given a market and our fair-value estimate, return (target_buy_yes_price,
    target_sell_yes_price). Inventory bias: if we're loaded long YES, widen
    our bid (lower) to discourage more buys; tighten our ask to encourage flatten."""
    fee = MM_CFG["kalshi_fee_per_contract"]
    offset_c = MM_CFG["quote_offset_cents"] / 100

    # Our buy-YES quote: just inside the existing yes_ask (we want to buy lower)
    target_buy = max(market["yes_bid"] + offset_c, fair_value - 0.5 * (market["yes_ask"] - market["yes_bid"]))
    # Our sell-YES quote: just inside the existing yes_bid (we want to sell higher)
    target_sell = min(market["yes_ask"] - offset_c, fair_value + 0.5 * (market["yes_ask"] - market["yes_bid"]))

    # Inventory skew: if long YES, lower our buy (less aggressive) and lower our sell (more aggressive flatten)
    inv_dollars = inventory * fair_value
    cap = MM_CFG["max_inventory_per_market"]
    if abs(inv_dollars) > 0.5 * cap:
        skew = (inv_dollars / cap) * 0.02  # up to 2c skew at full inventory
        target_buy  -= max(0, skew)
        target_sell -= max(0, skew)
        if inv_dollars < 0:
            # Short YES: opposite skew
            target_buy  += max(0, -skew)
            target_sell += max(0, -skew)

    # Quantize to cents and bound to [0.01, 0.99]
    target_buy  = max(0.01, min(0.99, round(target_buy,  2)))
    target_sell = max(0.01, min(0.99, round(target_sell, 2)))
    if target_sell - target_buy < 0.01:
        return None, None    # spread too tight, don't quote

    # Profitability check: spread must beat 2x fee
    if target_sell - target_buy < 2 * fee + 0.01:
        return None, None

    return target_buy, target_sell


# Demo on the widest-spread market
if markets:
    m = markets[0]
    fv = compute_fair_value(m)
    buy, sell = compute_target_quotes(m, fv, inventory=0)
    print(f"Demo quote on {m['ticker']}:")
    print(f"  market:  ${m['yes_bid']:.2f} / ${m['yes_ask']:.2f}  (spread {m['spread_cents']:.0f}c)")
    print(f"  fair:    ${fv:.3f}")
    print(f"  ours:    ${buy:.2f} (bid) / ${sell:.2f} (ask)" if buy is not None
          else "  not quotable (spread too tight or unprofitable)")


## 7. Inventory + fill tracking

In [ ]:
def get_inventory(ticker):
    conn = _mm_conn()
    row = conn.execute(
        "SELECT net_yes, total_cost FROM mm_inventory WHERE ticker=?", (ticker,)
    ).fetchone()
    conn.close()
    if row is None:
        return {"net_yes": 0, "total_cost": 0.0}
    return {"net_yes": row["net_yes"], "total_cost": float(row["total_cost"])}


def update_inventory(ticker, side, action, contracts, price):
    """Record a fill and update inventory."""
    inv = get_inventory(ticker)
    delta_yes = contracts if (side == "yes" and action == "buy") else \
                 -contracts if (side == "yes" and action == "sell") else \
                 -contracts if (side == "no" and action == "buy") else \
                 contracts  # short NO = long YES, etc.
    new_net = inv["net_yes"] + delta_yes
    cost_change = price * contracts if action == "buy" else -price * contracts
    new_total = inv["total_cost"] + cost_change

    conn = _mm_conn()
    conn.execute("""
        INSERT INTO mm_inventory(ticker, net_yes, total_cost, last_updated)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(ticker) DO UPDATE SET
            net_yes      = ?,
            total_cost   = ?,
            last_updated = ?
    """, (ticker, new_net, new_total, datetime.now(timezone.utc).isoformat(),
          new_net, new_total, datetime.now(timezone.utc).isoformat()))
    conn.commit(); conn.close()


def record_fill(ticker, side, action, contracts, price, quote_id=None):
    fees = MM_CFG["kalshi_fee_per_contract"] * contracts
    cost_dollars = price * contracts
    conn = _mm_conn()
    conn.execute("""
        INSERT INTO mm_fills(ts, ticker, side, action, price, contracts,
                              cost_dollars, fees_dollars, quote_id)
        VALUES (?,?,?,?,?,?,?,?,?)
    """, (datetime.now(timezone.utc).isoformat(), ticker, side, action,
          float(price), int(contracts), float(cost_dollars), float(fees),
          quote_id))
    conn.commit(); conn.close()
    update_inventory(ticker, side, action, contracts, price)


def total_inventory_dollars():
    conn = _mm_conn()
    row = conn.execute(
        "SELECT COALESCE(SUM(ABS(total_cost)), 0) FROM mm_inventory"
    ).fetchone()
    conn.close()
    return float(row[0])


## 8. Order placement + management

In [ ]:
def place_quote(ticker, side, price, contracts, fair_value, market_snapshot):
    """Place a single limit order. In paper mode, just record locally.
    Returns the local quote row id."""
    mode = MM_CFG.get("mode", "paper")
    order_id = None

    if mode == "live":
        if kalshi_live is None:
            print(f"  REFUSE live quote: kalshi_live not initialized")
            return None
        if not MM_CFG.get("i_acknowledge_real_money"):
            print(f"  REFUSE live quote: ack flag off")
            return None
        try:
            kwargs = {"yes_price": price} if side == "yes" else {"no_price": 1 - price}
            resp = kalshi_live.place_order(ticker, side=side, action="buy",
                                            count=contracts, type_="limit",
                                            **kwargs)
            order_id = resp.get("order", {}).get("order_id")
        except Exception as e:
            print(f"  live quote FAILED {ticker} {side} ${price:.2f}: {e}")
            return None

    conn = _mm_conn()
    conn.execute("""
        INSERT INTO mm_quotes(ts, ticker, side, price, contracts, order_id,
                                client_order_id, status, fair_value,
                                market_yes_bid, market_yes_ask)
        VALUES (?,?,?,?,?,?,?,?,?,?,?)
    """, (datetime.now(timezone.utc).isoformat(), ticker, side,
          float(price), int(contracts), order_id,
          f"mm-{int(time.time()*1000)}", "resting",
          float(fair_value), float(market_snapshot["yes_bid"]),
          float(market_snapshot["yes_ask"])))
    qid = conn.execute("SELECT last_insert_rowid()").fetchone()[0]
    conn.commit(); conn.close()
    print(f"  {'LIVE' if mode=='live' else 'PAPER'} QUOTE #{qid}  "
          f"{ticker:30s} {side:>3s} @ ${price:.2f} x{contracts}")
    return qid


def cancel_quote(quote_id):
    conn = _mm_conn()
    row = conn.execute(
        "SELECT order_id, status FROM mm_quotes WHERE id=?", (quote_id,)
    ).fetchone()
    if row is None or row["status"] != "resting":
        conn.close(); return False

    if MM_CFG["mode"] == "live" and row["order_id"] and kalshi_live:
        try:
            kalshi_live.cancel_order(row["order_id"])
        except Exception as e:
            print(f"  live cancel FAILED #{quote_id}: {e}")

    conn.execute("""
        UPDATE mm_quotes SET status='canceled', canceled_at=? WHERE id=?
    """, (datetime.now(timezone.utc).isoformat(), quote_id))
    conn.commit(); conn.close()
    return True


def cancel_all_resting(ticker=None):
    conn = _mm_conn()
    if ticker:
        rows = conn.execute(
            "SELECT id FROM mm_quotes WHERE ticker=? AND status='resting'", (ticker,)
        ).fetchall()
    else:
        rows = conn.execute(
            "SELECT id FROM mm_quotes WHERE status='resting'"
        ).fetchall()
    conn.close()
    for r in rows:
        cancel_quote(r["id"])
    return len(rows)


## 9. Paper-mode fill detection (simulation against live order book)

In [ ]:
def simulate_paper_fills():
    """For each resting paper quote, check if the live market\'s opposite-side
    has crossed our price. If so, mark it filled and update inventory."""
    if MM_CFG["mode"] != "paper":
        return 0
    conn = _mm_conn()
    quotes = conn.execute(
        "SELECT * FROM mm_quotes WHERE status='resting'"
    ).fetchall()
    conn.close()
    n_filled = 0
    for q in quotes:
        try:
            m = kalshi_md.get_market(q["ticker"]).get("market", {})
            yb = m.get("yes_bid"); ya = m.get("yes_ask")
            if yb is None or ya is None: continue
            yb, ya = float(yb)/100, float(ya)/100
            our_price = float(q["price"])
            our_side  = q["side"]   # we always 'buy' YES or 'buy' NO

            # YES side limit-buy fills if the live yes_ask drops to our price
            # NO  side limit-buy fills if the live no_ask  drops to (1 - our_price)
            filled = False
            if our_side == "yes" and ya <= our_price:
                filled = True
            elif our_side == "no":
                no_ask = 1 - yb
                if no_ask <= our_price:
                    filled = True
            if filled:
                conn = _mm_conn()
                conn.execute("""
                    UPDATE mm_quotes SET status='filled', filled_at=? WHERE id=?
                """, (datetime.now(timezone.utc).isoformat(), q["id"]))
                conn.commit(); conn.close()
                record_fill(q["ticker"], our_side, "buy",
                            q["contracts"], our_price, quote_id=q["id"])
                n_filled += 1
                print(f"  PAPER FILL #{q['id']}  {q['ticker']} {our_side} "
                      f"@ ${our_price:.2f} x{q['contracts']}")
        except Exception as e:
            print(f"  paper fill check err on quote #{q['id']}: {e}")
    return n_filled


## 10. Main loop — quote, refresh, fill, manage

In [ ]:
_MM_STATE = {"running": False, "thread": None, "log": []}


def mm_one_cycle():
    """Single pass: refresh markets, simulate/check fills, refresh quotes."""
    # 1. Check for paper fills
    if MM_CFG["mode"] == "paper":
        simulate_paper_fills()

    # 2. Find quotable markets
    markets = find_quotable_markets()
    if not markets:
        return
    # Take only the top-N widest spreads
    targets = markets[:5]

    # 3. For each target, ensure we have an active quote-pair
    conn = _mm_conn()
    resting = pd.read_sql_query(
        "SELECT ticker, side, id, price FROM mm_quotes WHERE status='resting'",
        conn)
    conn.close()
    resting_by_tk = {tk: g for tk, g in resting.groupby("ticker")}

    for m in targets:
        tk = m["ticker"]
        # Time exit
        if m["ttl_min"] <= MM_CFG["time_exit_min"]:
            cancelled = cancel_all_resting(tk)
            if cancelled:
                _MM_STATE["log"].append(
                    f"time-exit cancel {cancelled} on {tk} (ttl {m['ttl_min']:.1f}m)")
            continue

        inv = get_inventory(tk)
        fv = compute_fair_value(m)
        buy_p, sell_p = compute_target_quotes(m, fv, inventory=inv["net_yes"])
        if buy_p is None or sell_p is None:
            continue

        # Inventory caps: skip the side we're already loaded on
        inv_dollars = inv["net_yes"] * fv
        cap = MM_CFG["max_inventory_per_market"]

        existing = resting_by_tk.get(tk)
        existing_buy  = existing[existing["side"] == "yes"] if existing is not None else None
        existing_sell = existing[existing["side"] == "no"]  if existing is not None else None

        # Buy-YES quote (long-side entry)
        if inv_dollars < cap:
            need_new = (existing_buy is None or len(existing_buy) == 0
                        or abs(float(existing_buy.iloc[0]["price"]) - buy_p) >
                            MM_CFG["max_quote_drift_cents"]/100)
            if need_new:
                if existing_buy is not None and len(existing_buy):
                    cancel_quote(int(existing_buy.iloc[0]["id"]))
                if total_inventory_dollars() + buy_p * MM_CFG["default_size_contracts"] \
                   <= MM_CFG["max_inventory_total"]:
                    place_quote(tk, "yes", buy_p,
                                  MM_CFG["default_size_contracts"], fv, m)

        # Sell-YES (== buy-NO at 1-p) quote (flatten / short side)
        if inv_dollars > -cap:
            target_no_price = 1 - sell_p
            need_new = (existing_sell is None or len(existing_sell) == 0
                        or abs(float(existing_sell.iloc[0]["price"]) - target_no_price) >
                            MM_CFG["max_quote_drift_cents"]/100)
            if need_new:
                if existing_sell is not None and len(existing_sell):
                    cancel_quote(int(existing_sell.iloc[0]["id"]))
                if total_inventory_dollars() + target_no_price * MM_CFG["default_size_contracts"] \
                   <= MM_CFG["max_inventory_total"]:
                    place_quote(tk, "no", target_no_price,
                                  MM_CFG["default_size_contracts"], fv, m)


def _mm_worker():
    while _MM_STATE["running"]:
        try:
            mm_one_cycle()
        except Exception as e:
            _MM_STATE["log"].append(f"cycle err: {e}")
        slept = 0
        while slept < MM_CFG["loop_interval_sec"] and _MM_STATE["running"]:
            time.sleep(0.5); slept += 0.5


def start_market_maker():
    if _MM_STATE["running"]:
        print("market maker already running"); return
    _MM_STATE.update({"running": True, "thread": None, "log": []})
    th = threading.Thread(target=_mm_worker, daemon=True, name="kalshi_mm")
    th.start()
    _MM_STATE["thread"] = th
    print(f"market maker started in {MM_CFG['mode'].upper()} mode")
    print(f"  loop interval:    {MM_CFG['loop_interval_sec']}s")
    print(f"  refresh quotes:   every {MM_CFG['quote_refresh_sec']}s")
    print(f"  default size:     {MM_CFG['default_size_contracts']} contracts")
    print(f"  max inv / market: ${MM_CFG['max_inventory_per_market']}")
    print(f"  max inv total:    ${MM_CFG['max_inventory_total']}")


def stop_market_maker():
    _MM_STATE["running"] = False
    print("market maker stopping...")


## 11. Diagnostics + portfolio

In [ ]:
def mm_status(last_n=10):
    print("="*72)
    print(f"  MARKET MAKER STATUS @ {datetime.now(timezone.utc).isoformat()}")
    print("="*72)
    print(f"  running:   {_MM_STATE['running']}")
    print(f"  mode:      {MM_CFG['mode']}")

    conn = _mm_conn()
    n_resting = conn.execute(
        "SELECT COUNT(*) FROM mm_quotes WHERE status='resting'").fetchone()[0]
    n_filled = conn.execute(
        "SELECT COUNT(*) FROM mm_quotes WHERE status='filled'").fetchone()[0]
    n_canceled = conn.execute(
        "SELECT COUNT(*) FROM mm_quotes WHERE status='canceled'").fetchone()[0]
    inv = pd.read_sql_query("SELECT * FROM mm_inventory", conn)
    fills = pd.read_sql_query(
        "SELECT * FROM mm_fills ORDER BY ts DESC LIMIT ?", conn,
        params=[last_n])
    conn.close()

    print(f"\n  quotes:   resting={n_resting}  filled={n_filled}  canceled={n_canceled}")
    print(f"\n  inventory ({len(inv)} markets):")
    if len(inv):
        print(inv.to_string(index=False))

    if len(fills):
        print(f"\n  recent fills (last {len(fills)}):")
        cols = ["ts","ticker","side","action","price","contracts","cost_dollars"]
        print(fills[cols].to_string(index=False))


def mm_pnl():
    conn = _mm_conn()
    fills = pd.read_sql_query("SELECT * FROM mm_fills ORDER BY ts", conn)
    inv = pd.read_sql_query("SELECT * FROM mm_inventory", conn)
    conn.close()

    if len(fills) == 0:
        print("no fills yet."); return

    # Realized PnL: paired buys + sells on same ticker/side
    fills["signed_cost"] = np.where(fills["action"] == "buy",
                                       -fills["cost_dollars"],
                                       +fills["cost_dollars"])
    realized = fills.groupby("ticker")["signed_cost"].sum()
    fees = fills["fees_dollars"].sum()

    # Mark-to-market open inventory
    unrealized = 0.0
    for _, row in inv.iterrows():
        if row["net_yes"] == 0: continue
        try:
            m = kalshi_md.get_market(row["ticker"]).get("market", {})
            yb, ya = m.get("yes_bid"), m.get("yes_ask")
            if yb is None or ya is None: continue
            mid = (float(yb) + float(ya)) / 200
            unrealized += row["net_yes"] * mid - row["total_cost"]
        except Exception:
            pass

    total = realized.sum() + unrealized - fees
    print(f"  realized:     ${realized.sum():>+10.2f}")
    print(f"  unrealized:   ${unrealized:>+10.2f}")
    print(f"  fees paid:    ${fees:>+10.2f}")
    print(f"  net PnL:      ${total:>+10.2f}")
    if len(realized):
        print(f"\n  by ticker:")
        for tk, pnl in realized.items():
            print(f"    {tk:35s} ${pnl:>+10.2f}")


## 12. Run book

### Paper mode (recommended first)
```python
MM_CFG["mode"] = "paper"
start_market_maker()
mm_status()
mm_pnl()
```

The paper sim watches live Kalshi orderbooks and marks our quote as filled when the opposite side of the market crosses our price. It's optimistic (assumes guaranteed fill at our quoted price), but it's a useful sanity test.

### Live mode (real orders)
```python
MM_CFG["i_acknowledge_real_money"] = True
MM_CFG["mode"] = "live"
start_market_maker()
```

Live mode uses `kalshi_live.place_order(... type_="limit", ...)` and `cancel_order(...)`.

### Stop
```python
stop_market_maker()
cancel_all_resting()  # explicitly clean up any open orders
```

### Tune
```python
MM_CFG["min_spread_cents"]         = 4    # only quote markets with >= 4c spread
MM_CFG["default_size_contracts"]   = 30   # bigger quotes
MM_CFG["max_inventory_per_market"] = 100  # carry more inventory
MM_CFG["quote_refresh_sec"]        = 15   # more aggressive cancel/replace
```

### Risk knobs
- `session_loss_halt_usd`: halt new quotes once realized losses exceed this (per session)
- `max_inventory_per_market`: per-ticker $ at risk
- `max_inventory_total`: across all tickers
- `time_exit_min`: cancel + flatten in last N min before settlement
